# Engine Size, Vehicle Class, and Fuel Consumption in Canada
## Phase 2 - Data Wrangling

**Author:** Matthew Meede  
**Scope:** Model years 2015-2024

### Project purpose
This notebook prepares the Natural Resources Canada fuel consumption
dataset for analysis of the relationship between engine size, vehicle
class, and combined fuel consumption among gasoline vehicles.

The preparation includes inspecting the original data, documenting
cleaning decisions, grouping SUV class labels consistently, creating
derived metrics, and validating the final analytical dataset.

## Data Acquisition

The original dataset is Natural Resources Canada's 2015–2024 Fuel
Consumption Ratings CSV.

Source: https://open.canada.ca/data/dataset/98f1a129-f628-4ce4-b24d-6f16bf24dd64

The data is loaded without modifying the original file. The first five
rows are displayed below to show its initial structure.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Use the project CSV locally; use the source URL when it is absent.
data_path = Path("../Data/my2015-2024-fuel-consumption-ratings.csv")
source_url = (
    "https://open.canada.ca/data/dataset/"
    "98f1a129-f628-4ce4-b24d-6f16bf24dd64/resource/"
    "c98b9dc8-b23f-4cd8-8b19-e892da1e4688/download/"
    "my2015-2024-fuel-consumption-ratings.csv"
)
source = data_path if data_path.exists() else source_url
raw = pd.read_csv(source)
print("Loaded from:", source)
display(raw.head())

Loaded from: ..\Data\my2015-2024-fuel-consumption-ratings.csv


,Model year,Make,Model,Vehicle class,Engine size (L),Cylinders,Transmission,Fuel type,City (L/100 km),Highway (L/100 km),Combined (L/100 km),Combined (mpg),CO2 emissions (g/km),CO2 rating,Smog rating
0,2015,Acura,ILX,Compact,2.0,4,AS5,Z,9.7,6.7,8.3,34,191,NaN,NaN
1,2015,Acura,ILX,Compact,2.4,4,M6,Z,10.8,7.4,9.3,30,214,NaN,NaN
2,2015,Acura,ILX Hybrid,Compact,1.5,4,AV7,Z,6.0,6.1,6.1,46,140,NaN,NaN
3,2015,Acura,MDX SH-AWD,Sport utility vehicle: Small,3.5,6,AS6,Z,12.7,9.1,11.1,25,255,NaN,NaN
4,2015,Acura,RDX AWD,Sport utility vehicle: Small,3.5,6,AS6,Z,12.1,8.7,10.6,27,244,NaN,NaN


## Initial Exploration

This section examines the dataset dimensions, column types, missing
values, numerical summaries, and vehicle class and fuel type labels.
These checks will guide the cleaning decisions.

In [2]:
print(f"Rows: {raw.shape[0]:,}")
print(f"Columns: {raw.shape[1]}")

raw.info()

Rows: 10,060
Columns: 15
<class 'pandas.DataFrame'>
RangeIndex: 10060 entries, 0 to 10059
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Model year            10060 non-null  int64  
 1   Make                  10060 non-null  str    
 2   Model                 10060 non-null  str    
 3   Vehicle class         10060 non-null  str    
 4   Engine size (L)       10060 non-null  float64
 5   Cylinders             10060 non-null  int64  
 6   Transmission          10060 non-null  str    
 7   Fuel type             10060 non-null  str    
 8   City (L/100 km)       10060 non-null  float64
 9   Highway (L/100 km)    10060 non-null  float64
 10  Combined (L/100 km)   10060 non-null  float64
 11  Combined (mpg)        10060 non-null  int64  
 12  CO2 emissions (g/km)  10060 non-null  int64  
 13  CO2 rating            8932 non-null   float64
 14  Smog rating           7826 non-null   float64
dtypes: fl

In [3]:
display(
    raw.isna().sum()
    .rename("Missing values")
    .to_frame()
)

display(raw.describe())

display(raw["Vehicle class"].value_counts(dropna=False))
display(raw["Fuel type"].value_counts(dropna=False))

print("Exact duplicate rows:", raw.duplicated().sum())

,Missing values
Model year,0
Make,0
Model,0
Vehicle class,0
Engine size (L),0
Cylinders,0
Transmission,0
Fuel type,0
City (L/100 km),0
Highway (L/100 km),0


,Model year,Engine size (L),Cylinders,City (L/100 km),Highway (L/100 km),Combined (L/100 km),Combined (mpg),CO2 emissions (g/km),CO2 rating,Smog rating
count,10060.000000,10060.000000,10060.00000,10060.000000,10060.000000,10060.000000,10060.000000,10060.000000,8932.000000,7826.000000
mean,2019.242048,3.144901,5.61839,12.483052,9.142803,10.979712,27.436282,253.711332,4.617443,4.816509
std,2.834109,1.340010,1.87591,3.404483,2.178649,2.808468,7.316270,60.474917,1.574763,1.783793
min,2015.000000,0.900000,3.00000,4.000000,3.900000,4.000000,11.000000,94.000000,1.000000,1.000000
25%,2017.000000,2.000000,4.00000,10.100000,7.600000,9.000000,22.000000,210.000000,4.000000,3.000000
50%,2019.000000,3.000000,6.00000,12.100000,8.900000,10.700000,26.000000,250.000000,5.000000,5.000000
75%,2022.000000,3.700000,6.00000,14.500000,10.400000,12.700000,31.000000,292.000000,5.000000,6.000000
max,2024.000000,8.400000,16.00000,30.700000,20.900000,26.100000,71.000000,608.000000,10.000000,8.000000


Vehicle class
Sport utility vehicle: Small       1868
Mid-size                           1393
Sport utility vehicle: Standard    1231
Compact                            1135
Pickup truck: Standard              878
Subcompact                          850
Full-size                           779
Two-seater                          639
Minicompact                         460
Station wagon: Small                282
Pickup truck: Small                 220
Special purpose vehicle             103
Minivan                              95
Station wagon: Mid-size              82
Van: Passenger                       45
Name: count, dtype: int64

Fuel type
X    4756
Z    4739
E     328
D     236
N       1
Name: count, dtype: int64

Exact duplicate rows: 0


## Data Cleaning

The original dataset contains 10,060 rows and 15 columns, with no exact
duplicate rows. Model year, engine size, vehicle class, fuel type, and
combined fuel consumption have no missing values.

CO2 rating has 1,128 missing values and Smog rating has 2,234.
These optional ratings are not needed for this investigation, so they
will be excluded from the analytical dataset rather than filled with
invented values or used to remove otherwise complete records.

The analytical dataset will retain regular gasoline (X) and premium
gasoline (Z) records for model years 2015–2024. Gasoline hybrid records
remain included, consistent with the proposal. Other fuel types are
excluded to match the study scope.

The original dataset will remain unchanged.

In [4]:
# Work on a copy to preserve the original data.
clean = raw.copy()

# Remove surrounding whitespace from text columns and column headings.
clean.columns = clean.columns.str.strip()

text_columns = [
    "Make", "Model", "Vehicle class", "Transmission", "Fuel type"
]

for column in text_columns:
    clean[column] = clean[column].str.strip()

rows_before = len(clean)

# Select the study's model years and gasoline fuel types.
in_scope = (
    clean["Model year"].between(2015, 2024)
    & clean["Fuel type"].isin(["X", "Z"])
)

clean = clean.loc[in_scope].copy()

# Exclude optional ratings that are outside the research question.
clean = clean.drop(columns=["CO2 rating", "Smog rating"])

print(f"Original rows: {rows_before:,}")
print(f"Gasoline rows retained: {len(clean):,}")
print(f"Rows excluded by scope: {rows_before - len(clean):,}")

display(clean.head())

Original rows: 10,060
Gasoline rows retained: 9,495
Rows excluded by scope: 565


,Model year,Make,Model,Vehicle class,Engine size (L),Cylinders,Transmission,Fuel type,City (L/100 km),Highway (L/100 km),Combined (L/100 km),Combined (mpg),CO2 emissions (g/km)
0,2015,Acura,ILX,Compact,2.0,4,AS5,Z,9.7,6.7,8.3,34,191
1,2015,Acura,ILX,Compact,2.4,4,M6,Z,10.8,7.4,9.3,30,214
2,2015,Acura,ILX Hybrid,Compact,1.5,4,AV7,Z,6.0,6.1,6.1,46,140
3,2015,Acura,MDX SH-AWD,Sport utility vehicle: Small,3.5,6,AS6,Z,12.7,9.1,11.1,25,255
4,2015,Acura,RDX AWD,Sport utility vehicle: Small,3.5,6,AS6,Z,12.1,8.7,10.6,27,244


## Data Enrichment

### Vehicle class grouping

Both `Sport utility vehicle: Small` and `Sport utility vehicle: Standard` are grouped as **SUV**. Only the original `Compact` class becomes **Compact**; subcompact and minicompact vehicles are not included. All other classes become **Other** in the comparison column and remain available for broader analysis. The original Vehicle class column is preserved.

### Estimated annual fuel use

Estimated annual fuel use equals combined consumption (L/100 km) multiplied by 15,000 km and divided by 100. This fixed distance is an illustrative assumption, not observed annual driving. The calculation adds an interpretable measure but no independent evidence beyond the original consumption rating.

In [5]:
class_mapping = {
    "Sport utility vehicle: Small": "SUV",
    "Sport utility vehicle: Standard": "SUV",
    "Compact": "Compact"
}
clean["Comparison group"] = (
    clean["Vehicle class"].map(class_mapping).fillna("Other")
)
annual_distance_km = 15_000
clean["Estimated annual fuel use (L)"] = (
    clean["Combined (L/100 km)"] * annual_distance_km / 100
).round(1)

grouping_audit = (
    clean.groupby(["Vehicle class", "Comparison group"])
    .size().reset_index(name="Vehicle records")
)
display(grouping_audit)
display(clean["Comparison group"].value_counts().to_frame())

# Aggregation prepares a class-by-year table for later comparisons.
yearly_summary = (
    clean.groupby(["Model year", "Comparison group"])
    .agg(records=("Model", "size"),
         mean_consumption=("Combined (L/100 km)", "mean"),
         median_consumption=("Combined (L/100 km)", "median"))
    .reset_index()
)
display(yearly_summary.head(6))

,Vehicle class,Comparison group,Vehicle records
0,Compact,Compact,1107
1,Full-size,Other,737
2,Mid-size,Other,1355
3,Minicompact,Other,460
4,Minivan,Other,87
5,Pickup truck: Small,Other,186
6,Pickup truck: Standard,Other,674
7,Special purpose vehicle,Other,88
8,Sport utility vehicle: Small,SUV,1814
9,Sport utility vehicle: Standard,SUV,1107


,count
Comparison group,
Other,5467
SUV,2921
Compact,1107


,Model year,Comparison group,records,mean_consumption,median_consumption
0,2015,Compact,176,9.557955,9.0
1,2015,Other,612,10.837255,10.6
2,2015,SUV,246,11.248780,11.1
3,2016,Compact,176,9.376705,9.0
4,2016,Other,592,10.671959,10.6
5,2016,SUV,259,11.174903,10.9


## Validation Checks

The following checks verify completeness, duplicates after transformations, numeric types, finite positive measurements, model-year scope, gasoline codes, consistent text, and the exact SUV/compact mapping. The annual estimate is checked against its formula. Extreme consumption values are displayed for review rather than automatically deleted: an unusual value can represent a legitimate configuration.

The grouped yearly table counts configurations equally. It is not weighted by vehicle sales. No scaling or date parsing is needed for this preparation: model year is already an integer and consumption is already numeric in a consistent unit.

In [6]:
import numpy as np

measurement_columns = [
    "Engine size (L)", "Cylinders", "City (L/100 km)",
    "Highway (L/100 km)", "Combined (L/100 km)",
    "Combined (mpg)", "CO2 emissions (g/km)"
]
suv_classes = [
    "Sport utility vehicle: Small", "Sport utility vehicle: Standard"
]
expected_groups = clean["Vehicle class"].map(class_mapping).fillna("Other")
checks = {
    "No missing values": not clean.isna().any().any(),
    "No exact duplicates after cleaning": not clean.duplicated().any(),
    "Model years within scope": clean["Model year"].between(2015, 2024).all(),
    "Model year has integer type": pd.api.types.is_integer_dtype(clean["Model year"]),
    "Only gasoline codes": clean["Fuel type"].isin(["X", "Z"]).all(),
    "Measurements have numeric types": all(
        pd.api.types.is_numeric_dtype(clean[c]) for c in measurement_columns
    ),
    "Measurements finite and positive": (
        np.isfinite(clean[measurement_columns].to_numpy()).all()
        and clean[measurement_columns].gt(0).all().all()
    ),
    "Cylinder counts are whole numbers": clean["Cylinders"].mod(1).eq(0).all(),
    "Text nonblank and trimmed": all(
        clean[c].ne("").all() and clean[c].eq(clean[c].str.strip()).all()
        for c in text_columns
    ),
    "Group mapping matches policy": clean["Comparison group"].eq(expected_groups).all(),
    "All and only SUV labels grouped as SUV": (
        clean["Comparison group"].eq("SUV")
        .eq(clean["Vehicle class"].isin(suv_classes)).all()
    ),
    "All and only Compact labels grouped as Compact": (
        clean["Comparison group"].eq("Compact")
        .eq(clean["Vehicle class"].eq("Compact")).all()
    ),
    "SUV and Compact groups both present": {"SUV", "Compact"}.issubset(
        set(clean["Comparison group"])
    ),
    "Annual fuel calculation correct": np.allclose(
        clean["Estimated annual fuel use (L)"],
        clean["Combined (L/100 km)"] * annual_distance_km / 100,
        rtol=0, atol=0.051
    ),
    "Grouped counts reconcile": yearly_summary["records"].sum() == len(clean)
}
validation_report = pd.DataFrame({
    "Check": checks.keys(),
    "Result": ["PASS" if value else "FAIL" for value in checks.values()]
})
display(validation_report)
assert all(checks.values()), "Validation failed; review the report."

# Inspect the lowest and highest rated consumption configurations.
review_columns = [
    "Model year", "Make", "Model", "Vehicle class",
    "Engine size (L)", "Combined (L/100 km)"
]
display(clean.nsmallest(3, "Combined (L/100 km)")[review_columns])
display(clean.nlargest(3, "Combined (L/100 km)")[review_columns])

,Check,Result
0,No missing values,PASS
1,No exact duplicates after cleaning,PASS
2,Model years within scope,PASS
3,Model year has integer type,PASS
4,Only gasoline codes,PASS
5,Measurements have numeric types,PASS
6,Measurements finite and positive,PASS
7,Cylinder counts are whole numbers,PASS
8,Text nonblank and trimmed,PASS
9,Group mapping matches policy,PASS


,Model year,Make,Model,Vehicle class,Engine size (L),Combined (L/100 km)
6863,2021,Hyundai,IONIQ Blue,Full-size,1.6,4.0
7840,2022,Hyundai,IONIQ Blue,Full-size,1.6,4.0
2745,2017,Hyundai,IONIQ Blue,Full-size,1.6,4.1


,Model year,Make,Model,Vehicle class,Engine size (L),Combined (L/100 km)
6532,2021,Bugatti,Chiron Pur Sport,Two-seater,8.0,26.1
7510,2022,Bugatti,Chiron Pur Sport,Two-seater,8.0,26.1
7511,2022,Bugatti,Chiron Super Sport,Two-seater,8.0,26.1


## Final Analytical Dataset

All validation checks passed. The final dataset contains **9,495 gasoline configurations**: **2,921 SUVs**, **1,107 compact cars**, and **5,467 other configurations**. The full dataset retains all classes for broader analysis; a separate SUV/compact subset is prepared below for that specific comparison.

The original 10,060 rows were reduced only by the scope filter, excluding 565 non-gasoline records. Two optional rating columns were removed, and two derived columns were added, leaving 15 columns. The retained dataset has no missing values or exact duplicate rows. The original file and class labels remain unchanged.

The extreme-record review shows low consumption for Hyundai IONIQ Blue configurations and high consumption for Bugatti Chiron configurations. These differences are plausible and are retained; no arbitrary outlier removal is applied. Statistical influence will be evaluated during Phase 3.

The data is ready for scatter plots, class comparisons, correlation, and regression. Limits remain: standardized ratings are not actual driving observations, configurations are not weighted by sales, and conventional gasoline hybrids remain included. Annual fuel use assumes 15,000 km for every configuration. No hypothesis conclusion is drawn in this preparation phase.

In [7]:
analytical = clean.reset_index(drop=True).copy()
suv_compact = analytical.loc[
    analytical["Comparison group"].isin(["SUV", "Compact"])
].copy()

print(f"Final dataset: {analytical.shape[0]:,} rows, {analytical.shape[1]} columns")
print(f"SUV/compact comparison subset: {len(suv_compact):,} rows")
display(analytical.head())
display(analytical.dtypes.rename("Data type").to_frame())
display(analytical.isna().sum().rename("Missing values").to_frame())

Final dataset: 9,495 rows, 15 columns
SUV/compact comparison subset: 4,028 rows


,Model year,Make,Model,Vehicle class,Engine size (L),Cylinders,Transmission,Fuel type,City (L/100 km),Highway (L/100 km),Combined (L/100 km),Combined (mpg),CO2 emissions (g/km),Comparison group,Estimated annual fuel use (L)
0,2015,Acura,ILX,Compact,2.0,4,AS5,Z,9.7,6.7,8.3,34,191,Compact,1245.0
1,2015,Acura,ILX,Compact,2.4,4,M6,Z,10.8,7.4,9.3,30,214,Compact,1395.0
2,2015,Acura,ILX Hybrid,Compact,1.5,4,AV7,Z,6.0,6.1,6.1,46,140,Compact,915.0
3,2015,Acura,MDX SH-AWD,Sport utility vehicle: Small,3.5,6,AS6,Z,12.7,9.1,11.1,25,255,SUV,1665.0
4,2015,Acura,RDX AWD,Sport utility vehicle: Small,3.5,6,AS6,Z,12.1,8.7,10.6,27,244,SUV,1590.0


,Data type
Model year,int64
Make,str
Model,str
Vehicle class,str
Engine size (L),float64
Cylinders,int64
Transmission,str
Fuel type,str
City (L/100 km),float64
Highway (L/100 km),float64


,Missing values
Model year,0
Make,0
Model,0
Vehicle class,0
Engine size (L),0
Cylinders,0
Transmission,0
Fuel type,0
City (L/100 km),0
Highway (L/100 km),0


## Reproducibility and Sources

Run the notebook from top to bottom with Python, pandas, NumPy, and IPython installed. It first reads the CSV from the project's sibling Data folder. If that file is unavailable, it reads the same CSV directly from NRCan's public download URL; that fallback requires internet access. No dataset files are written to the school server.

The prepared DataFrames are named **analytical**, **suv_compact**, and **yearly_summary**. They can be regenerated by rerunning the notebook. The outputs saved in this notebook document this dataset snapshot; a future revision of the source could change counts.

- [Dataset source and supporting resources](https://open.canada.ca/data/dataset/98f1a129-f628-4ce4-b24d-6f16bf24dd64)
- [NRCan fuel codes and table definitions](https://prod-natural-resources.azure.cloud.nrcan-rncan.gc.ca/energy-efficiency/transportation-energy-efficiency/personal-vehicles/understanding-tables)
- [Open Government Licence Canada](https://open.canada.ca/en/open-government-licence-canada/)

Contains information licensed under the Open Government Licence – Canada.